# Person 1 — Exploring DEMO, MCQ and DIQ (NHANES 2017–2018)

**Goal:** understand the raw files before cleaning anything. We only *look* here — no data is changed.

| File | Topic | Columns we use |
| --- | --- | --- |
| `DEMO_J` | Demographics | `SEQN`, `RIDAGEYR` (age), `RIAGENDR` (sex), `DMDEDUC2` (education), `INDFMPIR` (income ratio) |
| `MCQ_J` | Medical conditions | `MCQ160B` heart failure, `MCQ160C` coronary heart disease, `MCQ160D` angina, `MCQ160E` heart attack, `MCQ160F` stroke |
| `DIQ_J` | Diabetes | `DIQ010` ever told you have diabetes |

Codebooks: [DEMO_J](https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/DEMO_J.htm) · [MCQ_J](https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/MCQ_J.htm) · [DIQ_J](https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/DIQ_J.htm)

## 1. Load the three files

In [1]:
from pathlib import Path
import pandas as pd

RAW = Path("..") / "data" / "raw" / "nhanes" / "2017-2018"

demo = pd.read_sas(RAW / "DEMO_J.xpt", format="xport")
mcq = pd.read_sas(RAW / "MCQ_J.xpt", format="xport")
diq = pd.read_sas(RAW / "DIQ_J.xpt", format="xport")

for name, df in [("DEMO", demo), ("MCQ", mcq), ("DIQ", diq)]:
    print(f"{name}: {df.shape[0]:,} people x {df.shape[1]} columns")

DEMO: 9,254 people x 46 columns
MCQ: 8,897 people x 76 columns
DIQ: 8,897 people x 54 columns


## 2. Is every person in each file only once? Who is missing?

In [2]:
for name, df in [("DEMO", demo), ("MCQ", mcq), ("DIQ", diq)]:
    print(f"{name}: SEQN unique = {df.SEQN.is_unique}")

missing_from_mcq = demo[~demo.SEQN.isin(mcq.SEQN)]
print()
print(f"In DEMO but not in MCQ/DIQ: {len(missing_from_mcq)} people")
print("Their ages:", missing_from_mcq.RIDAGEYR.round(2).value_counts().to_dict())

DEMO: SEQN unique = True
MCQ: SEQN unique = True
DIQ: SEQN unique = True

In DEMO but not in MCQ/DIQ: 357 people
Their ages: {0.0: 357}


## 3. Age (`RIDAGEYR`)

Watch out for two traps: age 0 is stored as a tiny number (`5.4e-79`, a SAS file quirk), and everyone 80 or older is recorded as 80.

In [3]:
age = demo.RIDAGEYR
print("Smallest raw value:", age.min())
print("Values that are really 0 (stored as ~5.4e-79):", (age.abs() < 1e-10).sum())
print("Largest value:", age.max())
print()
print("Adults aged 20+:", (age >= 20).sum())
print("Exactly 80 (= 80 or older):", (age == 80).sum())

Smallest raw value: 5.397605346934028e-79
Values that are really 0 (stored as ~5.4e-79): 357
Largest value: 80.0

Adults aged 20+: 5569
Exactly 80 (= 80 or older): 427


## 4. Sex (`RIAGENDR`) — 1 = male, 2 = female

In [4]:
demo.RIAGENDR.value_counts(dropna=False)

RIAGENDR
2.0    4697
1.0    4557
Name: count, dtype: int64

## 5. Education (`DMDEDUC2`) — adults 20+ only

1 = less than 9th grade · 2 = 9–11th grade · 3 = high school / GED · 4 = some college · 5 = college graduate · 7 = refused · 9 = don't know

In [5]:
adults = demo[demo.RIDAGEYR >= 20]
print("Under-20s with an education value:", demo[demo.RIDAGEYR < 20].DMDEDUC2.notna().sum())
adults.DMDEDUC2.value_counts(dropna=False).sort_index()

Under-20s with an education value: 0


DMDEDUC2
1.0     479
2.0     638
3.0    1325
4.0    1778
5.0    1336
7.0       2
9.0      11
Name: count, dtype: int64

## 6. Income-to-poverty ratio (`INDFMPIR`) — 0 to 5, where 5 means 5 or more

In [6]:
print("Adults with missing income ratio:", adults.INDFMPIR.isna().sum(), "of", len(adults))
print("People exactly at the cap of 5:", (demo.INDFMPIR == 5).sum())
adults.INDFMPIR.describe()

Adults with missing income ratio: 791 of 5569
People exactly at the cap of 5: 1208


count    4.778000e+03
mean     2.555019e+00
std      1.602175e+00
min      5.397605e-79
25%      1.200000e+00
50%      2.140000e+00
75%      4.080000e+00
max      5.000000e+00
Name: INDFMPIR, dtype: float64

## 7. Medical conditions and diabetes — the future target

We join the condition columns onto DEMO by `SEQN` (left join, so nobody is lost) to compare adults and under-20s.

MCQ codes: 1 = yes · 2 = no · 7 = refused · 9 = don't know · blank = not asked.
DIQ010 codes: 1 = yes · 2 = no · **3 = borderline** · 7 = refused · 9 = don't know.

In [7]:
conditions = {"MCQ160B": "heart failure", "MCQ160C": "coronary heart disease", "MCQ160D": "angina",
              "MCQ160E": "heart attack", "MCQ160F": "stroke"}

people = (demo[["SEQN", "RIDAGEYR"]]
          .merge(mcq[["SEQN", *conditions]], on="SEQN", how="left")
          .merge(diq[["SEQN", "DIQ010"]], on="SEQN", how="left"))
adults_c = people[people.RIDAGEYR >= 20]
kids_c = people[people.RIDAGEYR < 20]

rows = []
for col in [*conditions, "DIQ010"]:
    counts = adults_c[col].value_counts(dropna=False)
    rows.append({"column": col, "meaning": conditions.get(col, "diabetes"),
                 "yes (1)": counts.get(1.0, 0), "no (2)": counts.get(2.0, 0),
                 "borderline (3)": counts.get(3.0, 0), "don't know (9)": counts.get(9.0, 0),
                 "under-20s answered": kids_c[col].notna().sum()})
pd.DataFrame(rows)

,column,meaning,yes (1),no (2),borderline (3),don't know (9),under-20s answered
0,MCQ160B,heart failure,201,5351,0,17,0
1,MCQ160C,coronary heart disease,265,5288,0,16,0
2,MCQ160D,angina,161,5378,0,30,0
3,MCQ160E,heart attack,270,5289,0,10,0
4,MCQ160F,stroke,273,5286,0,10,0
5,DIQ010,diabetes,876,4517,172,4,3328


## 8. Preview: how many adults have at least one condition?

This is only a **preview** to understand the target. The final rule (especially for *borderline* and *don't know*) is decided and written in `people.py`.

In [8]:
cond_cols = list(conditions)
any_yes = (adults_c[cond_cols] == 1).any(axis=1) | (adults_c.DIQ010 == 1)
with_borderline = any_yes | (adults_c.DIQ010 == 3)

print(f"Adults 20+: {len(adults_c):,}")
print(f"At least one condition (borderline NOT counted): {any_yes.sum():,} ({any_yes.mean():.1%})")
print(f"At least one condition (borderline counted):     {with_borderline.sum():,} ({with_borderline.mean():.1%})")

Adults 20+: 5,569
At least one condition (borderline NOT counted): 1,295 (23.3%)
At least one condition (borderline counted):     1,444 (25.9%)


In [9]:
# Adults whose target is unclear: no "yes" anywhere, not borderline,
# but at least one answer is "don't know" (9) instead of "no" (2)
all_clear_no = (adults_c[cond_cols] == 2).all(axis=1) & (adults_c.DIQ010 == 2)
unclear = ~with_borderline & ~all_clear_no
print("Clearly no condition:", all_clear_no.sum())
print("Unclear target:", unclear.sum())

Clearly no condition: 4092
Unclear target: 33


## What I found

| # | Finding | What it means for cleaning |
| --- | --- | --- |
| 1 | DEMO has **9,254** people; MCQ and DIQ have **8,897**. `SEQN` is unique in all three files. | Join on `SEQN`; one row per person is guaranteed. |
| 2 | The **357** people in DEMO but not in MCQ/DIQ are all **age 0** (babies). | They disappear anyway when we keep adults only. |
| 3 | Age 0 is stored as **5.4e-79**, not 0 (357 values). | Snap tiny values to 0 before any comparison. |
| 4 | **5,569** adults are aged 20+. **427** people are exactly 80 — that means "80 or older". | Keep adults 20+; note the age cap as a limitation. |
| 5 | Sex: 4,557 male (1), 4,697 female (2). No missing. | Recode later as agreed in the contract. |
| 6 | Education (adults): 2 *refused* (7) and 11 *don't know* (9); blank for all under-20s. | Turn 7 and 9 into missing. |
| 7 | Income ratio missing for **791** of 5,569 adults (14%); 1,208 people sit at the cap of 5. | Leave missing for the model to impute; cap means "5 or more". |
| 8 | The heart questions (MCQ160B–F) are **blank for every under-20** and answered by all adults. | Confirms: keep adults 20+ only. |
| 9 | Each condition has **10–30 "don't know" (9)** answers among adults. | Treat 9 as missing, not as "no". |
| 10 | Diabetes `DIQ010` among adults: 876 yes, 4,517 no, **172 borderline (3)**, 4 don't know. | Decide: is borderline yes or no? |
| 11 | Preview: **1,295 adults (23.3%)** have at least one condition; **1,444 (25.9%)** if borderline counts. | Target is imbalanced (~1 in 4 positive). |
| 12 | 33 adults have no "yes" but some "don't know" answers, so their target is unclear. | Decide: drop them or count as 0. |

### Decisions to make next (for `people.py` and the decision log)

1. Borderline diabetes (172 adults): count as 1, as 0, or drop?
2. Unclear target (33 adults): drop or count as 0?
3. Education and condition codes 7 / 9 → missing.
4. Age 80 cap → keep as 80 and mention it as a limitation.